# Urban Safety — Classification: Feature Inspection & Pre-ML Analysis

Loads pre-computed segment features from `csv/features_all_boroughs_with_location_id_augmented.csv`, applies safety classification, exports labelled CSV, and explores feature distributions prior to ML training.

**The augmented input keeps the existing feature columns and adds `public_transport_proximity_m` plus `dominant_land_use_score`.**

**See `03_London_ClassificationML.ipynb` for model training.**

## Cell 1 — Imports and config

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set(rc={'figure.figsize': (8, 8)})
pd.set_option('display.float_format', '{:.3f}'.format)

# ── File paths ────────────────────────────────────────────────────────────
FEATURES_CACHE = os.path.join('..', 'csv', 'features_all_boroughs_with_location_id_augmented.csv')

# ── Feature columns ───────────────────────────────────────────────────────
FEATURE_COLS = ['lighting', 'visibility', 'connectivity', 'enclosure']

# ── Severity / safety config ──────────────────────────────────────────────
SEVERITY_ORDER   = ['Low', 'Medium', 'High']
SEVERITY_COLORS  = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
SEVERITY_WEIGHTS = {'Low': 1, 'Medium': 2, 'High': 3}

## Cell 2 — Load pre-computed features
Loads the cached feature CSV produced by `00_DataFetching.ipynb` and includes the two OSM-derived enrichment columns.

In [ ]:
features = pd.read_csv(FEATURES_CACHE)
print(f'\u2713 Loaded {len(features):,} segments from {FEATURES_CACHE}')
print(f'  Boroughs : {features["borough"].nunique()}')
print(features['borough'].value_counts().to_string())

## Cell 3 — Logarithmic crime score formula
Defined here for reference and validation. The actual scoring was applied during data fetch.

In [ ]:
def calculate_logarithmic_crime_score(petty_count, medium_count, serious_count):
    """
    score = [1.3^ln(petty+1) - 1] + [2.0^ln(medium+1) - 1] + [3.0^ln(serious+1) - 1]
    0 crimes -> score = 0. Serious crimes grow faster than petty crimes.
    """
    petty_score   = np.maximum(0, np.power(1.3, np.log(petty_count   + 1)) - 1)
    medium_score  = np.maximum(0, np.power(2.0, np.log(medium_count  + 1)) - 1)
    serious_score = np.maximum(0, np.power(3.0, np.log(serious_count + 1)) - 1)
    return petty_score + medium_score + serious_score

## Cell 4 — Segment safety classification
Categorizes streets into 4 safety classes based on fixed crime score thresholds:
Safe (0), Low (0–15), Medium (15–75), High (75+).

In [ ]:
# ── Fixed-score binning ──────────────────────────────────────────────────
zero_mask = features['crime_score'] == 0
non_zero_scores = features.loc[~zero_mask, 'crime_score']

threshold_low_med  = 15
threshold_med_high = 75

features['safety_class'] = 'safe'
features.loc[~zero_mask, 'safety_class'] = pd.cut(
    non_zero_scores,
    bins=[0, threshold_low_med, threshold_med_high, np.inf],
    labels=['low', 'medium', 'high'],
    include_lowest=True
)
features['safety_class'] = pd.Categorical(
    features['safety_class'],
    categories=['safe', 'low', 'medium', 'high'],
    ordered=True
)

print('Safety class thresholds (fixed-score bins):')
print(f'  safe   : 0 (zero crimes)')
print(f'  low    : 0 to {threshold_low_med:.0f}')
print(f'  medium : {threshold_low_med:.0f} to {threshold_med_high:.0f}')
print(f'  high   : {threshold_med_high:.0f}+')
print(f'\nActual distribution:')
for label in ['safe', 'low', 'medium', 'high']:
    mask = features['safety_class'] == label
    if mask.sum() > 0:
        min_val = features.loc[mask, 'crime_score'].min()
        max_val = features.loc[mask, 'crime_score'].max()
        pct = 100 * mask.sum() / len(features)
        print(f'  {label:10s} : {min_val:8.4f} to {max_val:8.4f} ({mask.sum():6,} segments, {pct:5.1f}%)')

In [ ]:
# Outlier detection: DISABLED (using all non-zero data)
print('Outlier detection: DISABLED (using all non-zero data, no filtering)')

## Cell 5 — Export labelled segment CSV

In [ ]:
features['location_id'] = features['borough'].str.extract(r'(\w+)')[0] + '_' + features['osmid'].astype(str)

segment_scores = features[['borough', 'lighting', 'visibility', 'connectivity', 'enclosure',
                            'crime_score', 'crime_count', 'safety_class']].copy()
segment_scores['location_id'] = features['location_id'].values

segment_scores.to_csv('csvFiles/segment_crime_scores_w-id.csv', index=False)
print(f'\u2713 Exported {len(segment_scores):,} segments')
print(f'\u2713 Saved to csvFiles/segment_crime_scores_w-id.csv')
display(segment_scores.head(10))

## Cell 6 — Class balance check
Verify that safe / low / medium / high are reasonably distributed before training.

In [ ]:
print('=== CLASS BALANCE CHECK ===\n')
overall = features['safety_class'].value_counts().sort_index()
print('Overall distribution:')
for label, count in overall.items():
    pct = 100 * count / len(features)
    bar = '\u2588' * int(pct / 2)
    print(f'  {label:10s}: {count:6,}  ({pct:5.1f}%)  {bar}')

print(f'\nPer-borough breakdown:')
borough_dist = (
    features.groupby('borough')['safety_class']
    .value_counts().unstack(fill_value=0)
)
borough_dist.index = [
    b.replace('London Borough of ', '').replace(', London, UK', '').replace('City of ', '')
    for b in borough_dist.index
]
print(borough_dist.to_string())

min_class_pct = 100 * overall.min() / len(features)
if min_class_pct < 10:
    print(f'\n\u26a0  Smallest class is only {min_class_pct:.1f}% - consider adjusting score thresholds.')
else:
    print(f'\n\u2713  All classes >= {min_class_pct:.1f}% - balance is acceptable for training.')

## Cell 7 — Crime score distribution plots

In [ ]:
os.makedirs('plots', exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(features['crime_score'], bins=80, color='steelblue', alpha=0.7, edgecolor='none', label='Histogram')
ax.set_xlabel('Crime Score')
ax.set_ylabel('Frequency (log scale)')
ax.set_yscale('log')
ax.set_title('Distribution of Crime Scores Across All Segments')
ax.axvline(threshold_low_med, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Low->Medium ({threshold_low_med:.2f})')
ax.axvline(threshold_med_high, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Medium->High ({threshold_med_high:.2f})')
ax.set_xlim([features['crime_score'].min() - 5, features['crime_score'].max() + 10])
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('plots/crime_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
features_copy = features.copy()
features_copy['borough_short'] = (features_copy['borough']
    .str.replace('London Borough of ', '').str.replace(', London, UK', '').str.replace('City of ', ''))
features_copy.boxplot(column='crime_score', by='borough_short', ax=ax)
ax.set_xlabel('Borough'); ax.set_ylabel('Crime Score')
ax.set_title('Crime Score Distribution by Borough')
plt.suptitle(''); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/crime_score_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
class_counts = features['safety_class'].value_counts().reindex(['safe', 'low', 'medium', 'high'])
colors_list = ['#C8E6C9', '#FFF9C4', '#FFE0B2', '#FFCCBC']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors_list, edgecolor='black',
                   linewidth=2, alpha=0.85)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height, f'{int(height):,}',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Number of Segments', fontsize=12, fontweight='bold')
axes[0].set_title('Segments by Safety Class', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')
axes[0].spines[['top', 'right']].set_visible(False)

non_zero = features[features['crime_score'] > 0].copy()
axes[1].hist(non_zero['crime_score'], bins=50, color='steelblue', edgecolor='black',
             linewidth=0.5, alpha=0.7)
axes[1].set_xlabel('Crime Score (non-zero only)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Segments', fontsize=12, fontweight='bold')
axes[1].set_title(f'Crime Score Distribution (excluding {(features["crime_score"] == 0).sum():,} zero-score segments)',
                  fontsize=12, fontweight='bold')
axes[1].axvline(threshold_low_med, color='red', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Low->Medium ({threshold_low_med:.2f})')
axes[1].axvline(threshold_med_high, color='darkred', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Medium->High ({threshold_med_high:.2f})')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y', linestyle='--')
axes[1].spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('plots/crime_score_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature EDA — inspect the data and how it correlates

Non-spatial exploration of the 4 per-segment features, colored by `safety_class`.

In [ ]:
feat_df = features[FEATURE_COLS].copy()
print(f'Feature dataframe: {feat_df.shape[0]:,} segments x {feat_df.shape[1]} features')
display(feat_df.head(10))
summary = feat_df.describe().T
summary['skew'] = feat_df.skew()
print('Summary (with skew):')
display(summary.round(3))

sns.set_theme(style='whitegrid')
plt.figure(figsize=(6, 5))
corr = features[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature correlation matrix (Pearson)')
plt.tight_layout()
plt.savefig('plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr.round(3))

In [ ]:
_order = ['safe', 'low', 'medium', 'high']
_pal = {'safe': '#C8E6C9', 'low': '#FFF9C4', 'medium': '#FFE0B2', 'high': '#FFCCBC'}
_df = features[FEATURE_COLS + ['safety_class']].copy()
_df['safety_class'] = _df['safety_class'].astype('object')

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, col in zip(axes.ravel(), FEATURE_COLS):
    plot_data = _df[_df[col] > 0].copy() if col == 'lighting' else _df.copy()
    title_suffix = ' (non-zero lamps only)' if col == 'lighting' else ''
    sns.violinplot(data=plot_data, x='safety_class', y=col, order=_order,
                   palette=_pal, legend=False, inner='quartile', cut=0, ax=ax)
    ax.set_title(f'{col} by safety class{title_suffix}'); ax.set_xlabel('')
plt.suptitle('Feature distributions by safety class - Violin plots', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig('plots/feature_distributions_violin.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, col in zip(axes.ravel(), FEATURE_COLS):
    plot_data = _df[_df[col] > 0].copy() if col == 'lighting' else _df.copy()
    title_suffix = ' (non-zero lamps only)' if col == 'lighting' else ''
    sns.stripplot(data=plot_data, x='safety_class', y=col, order=_order,
                  hue='safety_class', hue_order=_order, palette=_pal,
                  size=5, alpha=0.8, legend=False, ax=ax)
    ax.set_title(f'{col} by safety class{title_suffix}'); ax.set_xlabel('')
plt.suptitle('Feature distributions by safety class - Individual points', fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig('plots/feature_distributions_strip.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
_sample = _df.sample(n=min(2000, len(_df)), random_state=42)
pairplot = sns.pairplot(_sample, vars=FEATURE_COLS, hue='safety_class',
                        hue_order=_order, palette=_pal, corner=True,
                        plot_kws={'s': 12, 'alpha': 1.0})
pairplot.savefig('plots/feature_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from pandas.plotting import parallel_coordinates
_z = (features[FEATURE_COLS] - features[FEATURE_COLS].mean()) / features[FEATURE_COLS].std()
_z['safety_class'] = features['safety_class'].astype('object').values
_z = _z.dropna(subset=['safety_class']).sample(n=min(800, len(_z)), random_state=42)
_present = [c for c in _order if c in _z['safety_class'].unique()]
fig, ax = plt.subplots(figsize=(11, 6))
parallel_coordinates(_z[FEATURE_COLS + ['safety_class']], 'safety_class',
                     color=[_pal[c] for c in _present], alpha=0.5, ax=ax)
ax.set_title('Parallel coordinates by safety class (z-scored, sampled)')
ax.set_ylabel('crime score (z)')
plt.tight_layout()
plt.savefig('plots/parallel_coordinates.png', dpi=150, bbox_inches='tight')
plt.show()

## Exponential Crime Score Validation

Demonstrates the scoring formula with example crime configurations.

In [ ]:
print('=== Exponential Crime Score Examples (base^ln(count+1)) ===\n')
print('Formula: score = [1.3^ln(petty+1) - 1] + [2.0^ln(medium+1) - 1] + [3.0^ln(serious+1) - 1]\n')

examples = [
    ('0 crimes', 0, 0, 0),
    ('1 Petty crime', 1, 0, 0),
    ('5 Petty crimes', 5, 0, 0),
    ('10 Petty crimes', 10, 0, 0),
    ('50 Petty crimes', 50, 0, 0),
    ('1 Medium crime', 0, 1, 0),
    ('5 Medium crimes', 0, 5, 0),
    ('20 Medium crimes', 0, 20, 0),
    ('1 Serious crime', 0, 0, 1),
    ('5 Serious crimes', 0, 0, 5),
    ('20 Serious crimes', 0, 0, 20),
    ('1L + 1M + 1S', 1, 1, 1),
    ('10L + 5M + 3S', 10, 5, 3),
    ('50L + 20M + 10S', 50, 20, 10),
]

for desc, p, m, s in examples:
    score = calculate_logarithmic_crime_score(p, m, s)
    print(f'{desc:25s} -> score = {score:8.4f}')

print('\n=== Score Distribution Statistics ===\n')
print(f'Min crime_score: {features["crime_score"].min():.4f}')
print(f'Max crime_score: {features["crime_score"].max():.4f}')
print(f'Mean crime_score: {features["crime_score"].mean():.4f}')
print(f'Median crime_score: {features["crime_score"].median():.4f}')
print(f'\nSafety class distribution:')
print(features['safety_class'].value_counts().sort_index())